# 174 — World models y simulación interna

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución 1 — Roles

a) **V** (encoder: observación → latente). b) **M** (dinámica latente
condicionada a la acción). c) **C** (política/controlador). d) **M** — la
MDN-RNN produce una *distribución* (mezcla de gaussianas) sobre el siguiente
latente, exactamente para representar futuros múltiples.


In [ ]:
roles = {"a": "V", "b": "M", "c": "C", "d": "M"}
print(roles)


## Solución 2 — Imaginación

a) [avanzar, saltar]: paso 1 desde A → B (0.9) o A (0.1), sin recompensa.
Paso 2 con saltar: desde B, P(C)=0.5 → 0.9·0.5 = 0.45; desde A, P(C)=0.4 →
0.1·0.4 = 0.04. **Retorno = 0.49**.

b) Sigue ganando [avanzar, avanzar] (0.72 > 0.64 > 0.49 > [saltar, avanzar]).

c) Ver código: enumeración exacta de trayectorias con recompensa a la primera
llegada a C.


In [ ]:
T = {
    "avanzar": {"A": {"B": 0.9, "A": 0.1}, "B": {"C": 0.8, "B": 0.2}, "C": {"C": 1.0}},
    "saltar":  {"A": {"C": 0.4, "A": 0.6}, "B": {"C": 0.5, "B": 0.5}, "C": {"C": 1.0}},
}
def retorno_esperado(plan, estado="A"):
    if not plan:
        return 0.0
    total = 0.0
    for destino, p in T[plan[0]][estado].items():
        if destino == "C" and estado != "C":
            total += p  # recompensa al entrar en C
        else:
            total += p * retorno_esperado(plan[1:], destino)
    return total

from itertools import product
for plan in product(["avanzar", "saltar"], repeat=2):
    print(plan, round(retorno_esperado(list(plan)), 4))
assert abs(retorno_esperado(["avanzar", "saltar"]) - 0.49) < 1e-9


## Solución 3 — Explotación del modelo

a) Con la transición real B→C = 0.5: retorno real de [avanzar, avanzar] =
0.9·0.5 = **0.45**, frente al 0.72 imaginado (sobreestimación del 60 %).

b) En el mundo real, [saltar, saltar] rinde 0.4 + 0.6·0.4 = **0.64** (no depende
de la transición errónea): habría sido el mejor plan.

c) Mitigaciones: (1) **Re-planificación frecuente**: tras ejecutar "avanzar" y
llegar a B, el agente observaría fallos repetidos de B→C, actualizaría el modelo
y cambiaría de plan — el error no se paga entero. (2) **Penalizar incertidumbre
con ensembles**: si varios modelos discrepan sobre B→C (0.8 vs 0.5), el plan que
depende de esa transición se penaliza y [saltar, saltar], menos sensible al
error, ganaría la comparación.


## Solución 4 — Afirmaciones de frontera (referencia)

Ejemplo de respuesta bien calibrada:


In [ ]:
result = run_lab("frontier", seed=174)
mis_claims = [
    {"claim": "DreamerV3 resuelve 150+ tareas con hiperparámetros fijos",
     "evidence": "arXiv:2301.04104, resultados replicados con código público",
     "maturity": "current"},
    {"claim": "el entrenamiento 100% en sueño transfiere a cualquier dominio",
     "evidence": "demostrado en CarRacing/VizDoom (arXiv:1803.10122), no en general",
     "maturity": "needs_replication"},
    {"claim": "los generadores de video actuales entienden la física",
     "evidence": None,
     "maturity": "unverified"},
]
for c in mis_claims:
    print(f"[{c['maturity']:>18}] {c['claim']}")
# Criterio: 'current' exige fuente + replicación; una demo en 2 entornos no
# generaliza (needs_replication); calidad visual no es evidencia de dinámica
# correcta ante intervenciones (unverified).
